# Terraform Generator avec Callbacks LangChain Natifs

Ce notebook démontre l'utilisation des **callbacks LangChain** pour tracker les phases d'exécution de manière native et élégante.

## Approche

Au lieu d'un wrapper superficiel, on utilise le système de callbacks natif de LangChain :
- **`on_tool_start`** : Détecte le début d'une phase (planning, validation, review)
- **`on_tool_end`** : Capture le résultat et le succès
- **`on_llm_start`** : Détecte la génération de code

## Avantages

✅ **Natif LangChain** : Utilise le système de callbacks standard
✅ **Non intrusif** : Pas de modification du workflow existant
✅ **Phases réelles** : Basées sur les appels d'outils, pas sur du parsing
✅ **Structuré** : Logs et rapports JSON propres
✅ **Extensible** : Facile d'ajouter des checks personnalisés

## Étape 1: Imports

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import json

# Load environment variables
load_dotenv()

# Setup path
import sys
sys.path.insert(0, str(Path.cwd().parent))

from terraform_agent import (
    Config,
    PromptManager,
    KnowledgeBase,
    TerraformGenerator,
    TerraformPhaseCallback,        # Simple phase tracking
    DetailedTerraformCallback,      # With security/BP checks
)

print("✓ Imports loaded successfully")

✓ Imports loaded successfully


## Étape 2: Configuration

In [2]:
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%H:%M:%S'
)

print("✓ Logging configured")

✓ Logging configured


## Étape 3: Initialisation des Composants

In [3]:
# Get project root
project_root = Path.cwd().parent

# Initialize components
config = Config(base_dir=project_root)
print(f"Project Root: {config.PROJECT_ROOT}")
print(f"Work Dir: {config.WORK_DIR}")

prompts = PromptManager(config)
print("\n✓ Prompts loaded")

knowledge_base = KnowledgeBase(config)

# Initialize generator (standard)
generator = TerraformGenerator(config, prompts, knowledge_base)

print("\n✓ Generator ready")

13:13:29 - terraform_agent.knowledge_base - INFO - Clearing 96 existing documents from vectorstore


Project Root: /Users/melkouhen/audit-tools/test-langchain
Work Dir: /Users/melkouhen/audit-tools/test-langchain/work

✓ Prompts loaded
📚 Loading knowledge base...
  ✓ Loaded 9 document(s) from /Users/melkouhen/audit-tools/test-langchain/rules
  ✓ Split into 96 chunks
  Creating new vectorstore database...
  🗑️ Clearing 96 existing documents...
  Indexing 96 documents...


13:13:31 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
13:13:31 - terraform_agent.tools - INFO - Initializing tools with config, prompts, and knowledge base
13:13:31 - terraform_agent.tools - INFO - Tools initialized - Review model: qwen2.5-coder:7b-instruct
13:13:31 - terraform_agent.tools - INFO - Model router enabled - Ollama tasks: {'review', 'summarization', 'parsing'}


  ✓ Vectorstore created and indexed
🤖 Setting up agent...
  ✓ System prompt loaded (7406 chars)
  ✓ User prompt loaded (1035 chars)
  ✓ Agent created with tools:
    - load_module_spec (module specifications)
    - search_knowledge_base (best practices)
    - terraform_init (initialize working directory)
    - terraform_validate (validate configuration)
    - terraform_plan (preview changes)
    - review_and_fix_code (code review)
  ✓ Model routing:
    - Ollama tasks: review, summarization, parsing
      • review: qwen2.5-coder:14b
      • summarization: qwen3.5:9b
      • parsing: qwen2.5-coder:7b-instruct

✓ Generator ready


## Étape 4: Exécution avec Callback Simple

Le callback `TerraformPhaseCallback` détecte automatiquement les phases :
- **PLANNING** : Quand `search_knowledge_base` ou `load_module_spec` est appelé
- **GENERATION** : Quand le LLM est invoqué
- **VALIDATION** : Quand `terraform_init`, `terraform_validate` ou `terraform_plan` est appelé
- **CODE_REVIEW** : Quand `review_and_fix_code` est appelé

In [4]:
# Create callback
callback = TerraformPhaseCallback(verbose=True)

# Load user prompt
user_prompt = (project_root / "user_prompts/1-bucket.md").read_text()

# Execute with callback
print("\n🚀 Starting Terraform generation with phase tracking...\n")
result = generator.run(user_prompt=user_prompt, callbacks=[callback])

print("\n✓ Execution completed")

13:13:31 - terraform_agent.callbacks - INFO - Phase started: GENERATION (triggered by llm)



🚀 Starting Terraform generation with phase tracking...

🛠️  Preparing workspace...

🚀 Starting Terraform Code Generation Agent
Timestamp: 2026-05-12 13:13:31

📝 Agent is running...
    (Agent will autonomously call: terraform_init, terraform_validate, terraform_plan, review_and_fix_code)
--------------------------------------------------------------------------------

🔧 PHASE: GENERATION


13:13:34 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


   → Calling write_todos
   ✅ write_todos completed


13:13:36 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:13:36 - terraform_agent.callbacks - INFO - Phase ended: GENERATION (4.88s)
13:13:36 - terraform_agent.callbacks - INFO - Phase ended: GENERATION (4.88s)
13:13:36 - terraform_agent.callbacks - INFO - Phase started: PLANNING (triggered by search_knowledge_base)
13:13:36 - terraform_agent.tools - INFO - Searching knowledge base for: structure google_storage_bucket
13:13:36 - terraform_agent.callbacks - INFO - Phase started: PLANNING (triggered by search_knowledge_base)
13:13:36 - terraform_agent.tools - INFO - Searching knowledge base for: isolation environnements dev prod Terraform
13:13:36 - terraform_agent.tools - INFO - Searching knowledge base for: backend state Terraform GCS
13:13:36 - terraform_agent.tools - INFO - Searching knowledge base for: sécurité google_storage_bucket
13:13:36 - terraform_agent.knowledge_base - INFO - Searching vectorstore for: 'structure google_storage_bu

...

   → Calling write_file
   ✅ write_file completed


13:16:57 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


   → Calling write_todos
   ✅ write_todos completed


13:17:09 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:17:09 - terraform_agent.generator - INFO - Agent execution completed successfully
13:17:09 - terraform_agent.callbacks - INFO - Phase ended: VALIDATION (120.20s)



   Phase completed in 120.20s

📊 EXECUTION SUMMARY

Phase                Duration        Status    
--------------------------------------------------
GENERATION           4.88s           ✅         
GENERATION           4.88s           ✅         
PLANNING             25.80s          ✅         
VALIDATION           7.85s           ✅         
CODE_REVIEW          59.28s          ✅         
CODE_REVIEW          59.29s          ✅         
VALIDATION           120.20s         ✅         

Total                218.08s

🔧 Tool Execution Summary:
   ✅ write_todos
   ❌ search_knowledge_base
   ✅ ls
   ✅ read_file
   ❌ terraform_init
   ❌ terraform_validate
   ❌ terraform_plan
   ✅ review_and_fix_code
   ✅ write_file

✅ SUCCESS
--------------------------------------------------------------------------------

✓ Execution completed


## Étape 5: Récupérer le Rapport

Le callback génère un rapport JSON structuré avec :
- Log des phases (nom, durée, timestamp)
- Résultats des outils (succès/échec)
- Durée totale

In [5]:
# Get structured report
report = callback.get_report()

print("\n📊 Execution Report:")
print(json.dumps(report, indent=2))


📊 Execution Report:
{
  "execution_log": [
    {
      "phase": "GENERATION",
      "duration_seconds": 4.88,
      "timestamp": "2026-05-12T13:13:36.311228"
    },
    {
      "phase": "GENERATION",
      "duration_seconds": 4.88,
      "timestamp": "2026-05-12T13:13:36.313462"
    },
    {
      "phase": "PLANNING",
      "duration_seconds": 25.8,
      "timestamp": "2026-05-12T13:14:02.120857"
    },
    {
      "phase": "VALIDATION",
      "duration_seconds": 7.85,
      "timestamp": "2026-05-12T13:14:09.974710"
    },
    {
      "phase": "CODE_REVIEW",
      "duration_seconds": 59.28,
      "timestamp": "2026-05-12T13:15:09.259425"
    },
    {
      "phase": "CODE_REVIEW",
      "duration_seconds": 59.29,
      "timestamp": "2026-05-12T13:15:09.261020"
    },
    {
      "phase": "VALIDATION",
      "duration_seconds": 120.2,
      "timestamp": "2026-05-12T13:17:09.458936"
    }
  ],
  "tool_results": {
    "write_todos": {
      "output": "Command(update={'todos': [{'content':

## Étape 6: Exécution avec Callback Détaillé

Le callback `DetailedTerraformCallback` ajoute :
- **Security checks** : UBLA, encryption, public access, versioning, lifecycle
- **Best practices checks** : Module structure, variables, outputs, documentation
- **Scoring** : Security score et BP score (0-100%)

In [6]:
# Create detailed callback
detailed_callback = DetailedTerraformCallback(verbose=True)

# Execute with detailed callback
print("\n🚀 Starting Terraform generation with detailed tracking...\n")
result = generator.run(user_prompt=user_prompt, callbacks=[detailed_callback])

print("\n✓ Execution completed")

13:17:09 - terraform_agent.callbacks - INFO - Phase started: GENERATION (triggered by llm)



🚀 Starting Terraform generation with detailed tracking...

🛠️  Preparing workspace...

🚀 Starting Terraform Code Generation Agent
Timestamp: 2026-05-12 13:17:09

📝 Agent is running...
    (Agent will autonomously call: terraform_init, terraform_validate, terraform_plan, review_and_fix_code)
--------------------------------------------------------------------------------

🔧 PHASE: GENERATION


13:17:11 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:17:11 - terraform_agent.callbacks - INFO - Phase ended: GENERATION (2.23s)
13:17:11 - terraform_agent.callbacks - INFO - Phase started: PLANNING (triggered by search_knowledge_base)
13:17:11 - terraform_agent.callbacks - INFO - Phase ended: GENERATION (2.23s)
13:17:11 - terraform_agent.tools - INFO - Searching knowledge base for: structure environnements dev prod isolation
13:17:11 - terraform_agent.tools - INFO - Searching knowledge base for: nommage google_storage_bucket
13:17:11 - terraform_agent.callbacks - INFO - Phase started: PLANNING (triggered by search_knowledge_base)
13:17:11 - terraform_agent.knowledge_base - INFO - Searching vectorstore for: 'structure environnements dev prod isolation' (k=3)
13:17:11 - terraform_agent.knowledge_base - INFO - Searching vectorstore for: 'nommage google_storage_bucket' (k=3)
13:17:11 - terraform_agent.tools - INFO - Searching knowledge bas

   → Calling ls

   Phase completed in 2.23s

   Phase completed in 2.23s

📋 PHASE: PLANNING
   → Calling search_knowledge_base
   → Calling search_knowledge_base
   ✅ ls completed

📋 PHASE: PLANNING
   → Calling search_knowledge_base
   ✅ search_knowledge_base completed
   ✅ search_knowledge_base completed
   ✅ search_knowledge_base completed


13:17:15 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


...


   Phase completed in 335.26s

🔍 PHASE: CODE_REVIEW
   → Calling review_and_fix_code


13:27:38 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
13:27:38 - terraform_agent.knowledge_base - INFO - Found 0 results (0 chars) - preview: (empty)...
13:27:38 - terraform_agent.tools - INFO - Found 7 Terraform files
13:27:38 - terraform_agent.tools - INFO - Total code to review: 3497 bytes across 7 files
13:27:38 - terraform_agent.tools - INFO - Running code review with qwen2.5-coder:7b-instruct
13:27:45 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
13:28:55 - terraform_agent.tools - INFO - Code review completed in 76.62s, response length: 6676 bytes
13:28:55 - terraform_agent.tools - INFO - Code is compliant with best practices
13:28:55 - terraform_agent.tools - INFO - Code review and fix completed in 77.58s


   ✅ review_and_fix_code completed


13:29:03 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:29:03 - terraform_agent.generator - INFO - Agent execution completed successfully
13:29:03 - terraform_agent.callbacks - INFO - Phase ended: CODE_REVIEW (85.49s)



   Phase completed in 85.49s

📊 EXECUTION SUMMARY

Phase                Duration        Status    
--------------------------------------------------
GENERATION           2.23s           ✅         
GENERATION           2.23s           ✅         
PLANNING             23.35s          ✅         
VALIDATION           135.76s         ✅         
CODE_REVIEW          131.65s         ✅         
CODE_REVIEW          131.65s         ✅         
VALIDATION           335.26s         ✅         
CODE_REVIEW          85.49s          ✅         

Total                713.80s

🔧 Tool Execution Summary:
   ✅ ls
   ✅ search_knowledge_base
   ✅ write_todos
   ✅ write_file
   ❌ terraform_init
   ✅ edit_file
   ✅ terraform_validate
   ✅ read_file
   ✅ grep
   ❌ terraform_plan
   ✅ review_and_fix_code
   ✅ glob


🔒 Security Checks:
   ❌ UBLA
   ✅ Public Access Prevention
   ✅ Encryption
   ✅ Versioning
   ✅ Lifecycle Policies

   Security Score: 80%

📐 Best Practices Checks:
   ✅ Module Structure
   ✅ Variabl

## Étape 7: Rapport Détaillé avec Scores

In [7]:
# Get detailed report with security/BP checks
detailed_report = detailed_callback.get_report()

print("\n📊 Detailed Execution Report:")
print(json.dumps(detailed_report, indent=2))

# Extract scores
if "security_score" in detailed_report:
    print(f"\n🔒 Security Score: {detailed_report['security_score']:.0f}%")
    
if "bp_score" in detailed_report:
    print(f"📐 Best Practices Score: {detailed_report['bp_score']:.0f}%")


📊 Detailed Execution Report:
{
  "execution_log": [
    {
      "phase": "GENERATION",
      "duration_seconds": 2.23,
      "timestamp": "2026-05-12T13:17:11.772932"
    },
    {
      "phase": "GENERATION",
      "duration_seconds": 2.23,
      "timestamp": "2026-05-12T13:17:11.773992"
    },
    {
      "phase": "PLANNING",
      "duration_seconds": 23.35,
      "timestamp": "2026-05-12T13:17:35.128112"
    },
    {
      "phase": "VALIDATION",
      "duration_seconds": 135.76,
      "timestamp": "2026-05-12T13:19:50.886044"
    },
    {
      "phase": "CODE_REVIEW",
      "duration_seconds": 131.65,
      "timestamp": "2026-05-12T13:22:02.537310"
    },
    {
      "phase": "CODE_REVIEW",
      "duration_seconds": 131.65,
      "timestamp": "2026-05-12T13:22:02.539016"
    },
    {
      "phase": "VALIDATION",
      "duration_seconds": 335.26,
      "timestamp": "2026-05-12T13:27:37.799246"
    },
    {
      "phase": "CODE_REVIEW",
      "duration_seconds": 85.49,
      "timestam

## Étape 8: Créer un Callback Personnalisé

Vous pouvez facilement étendre le callback pour vos besoins spécifiques :

In [8]:
from terraform_agent.callbacks import DetailedTerraformCallback

class CustomCallback(DetailedTerraformCallback):
    """Custom callback with additional checks."""
    
    def on_tool_end(self, output: str, **kwargs):
        super().on_tool_end(output, **kwargs)
        
        tool_name = kwargs.get("name", "unknown")
        
        # Add custom logic
        if tool_name == "terraform_plan":
            if "to add" in output:
                print("   💡 Resources will be created")

# Use custom callback
custom_callback = CustomCallback(verbose=True)
print("\n✓ Custom callback ready for use")


✓ Custom callback ready for use


## Comparaison des Approches

### Avant (sans callback)
```python
result = generator.run(user_prompt)
# Output: logs bruts de l'agent
```

### Avec callback simple
```python
callback = TerraformPhaseCallback(verbose=True)
result = generator.run(user_prompt, callbacks=[callback])
# Output: Phases explicites + durées
```

### Avec callback détaillé
```python
callback = DetailedTerraformCallback(verbose=True)
result = generator.run(user_prompt, callbacks=[callback])
# Output: Phases + checks sécurité/BP + scores
report = callback.get_report()
```

## Avantages de l'Approche Callbacks

| Aspect | Pipeline Wrapper | Callbacks LangChain |
|--------|------------------|---------------------|
| **Architecture** | Wrapper superficiel | ✅ Natif LangChain |
| **Détection phases** | Parsing de texte | ✅ Hooks d'outils réels |
| **Intrusivité** | Nouveau composant | ✅ Option sur Generator existant |
| **Extensibilité** | Hériter du wrapper | ✅ Hériter du callback |
| **Maintenance** | Code dupliqué | ✅ Composant indépendant |
| **Performance** | Overhead | ✅ Minimal |

## Cas d'Usage

### 1. Développement/Debug
```python
callback = TerraformPhaseCallback(verbose=True)
generator.run(prompt, callbacks=[callback])
```

### 2. CI/CD avec Checks
```python
callback = DetailedTerraformCallback(verbose=False)
generator.run(prompt, callbacks=[callback])
report = callback.get_report()

if report["security_score"] < 80:
    raise Exception("Security score too low")
```

### 3. Logging Structuré
```python
import json
callback = DetailedTerraformCallback(verbose=False)
generator.run(prompt, callbacks=[callback])

# Save to file
report_file = f"logs/run-{datetime.now():%Y%m%d-%H%M%S}.json"
Path(report_file).write_text(json.dumps(callback.get_report(), indent=2))
```